# Chipathon 2026 — run your own RTL through the workshop padring

This is the notebook you want if you have the workshop-padring
template already prepared (either because you ran
`rtl2gds_chipathon_padring.ipynb` once, or because the chipathon
organisers handed you a zip/tarball with everything already
integrated). The goal here is the shortest possible path between
"I have my own Verilog" and "I have a fab-ready GDS that respects
the shuttle padring."

**What the padring gives you (do not modify).**
The `chip_top.sv` of the template is fixed and wires the padring to
your core via these ports (parameters come from the SLOT_WORKSHOP
block in `src/slot_defines.svh`):

| port                          | width | direction | notes |
|-------------------------------|-------|-----------|-------|
| `clk`                         | 1     | input     | from the Schmitt-trigger clk_pad |
| `rst_n`                       | 1     | input     | active-low reset (from rst_n_pad) |
| `input_in`                    | 1     | input     | spare CMOS input (inputs[0].pad) |
| `input_pu`, `input_pd`        | 1 ea  | output    | tie to `'0`; no pull-ups/downs |
| `bidir_in[19:0]`              | 20    | input     | from the 20 config pads (bi_24t) |
| `bidir_out[19:0]`             | 20    | output    | drives the 20 config pads |
| `bidir_oe/cs/sl/ie/pu/pd[19:0]` | 20 each | output | pad-cell controls; tied defaults work |
| `analog[59:0]`                | 60    | inout     | 60 analog pads (asig_5p0, 5 V) |

**Your contract:**
- Produce a `module chip_core (...)` with the signature shown above.
- Drive `bidir_out` with whatever you want visible on the 20
  digital output pins.
- Either tie off or use `analog[]` in your custom analog IP; the
  digital flow leaves them as pure pass-through to the pad cells.

Default-safe controls for the pads (copy into every `chip_core`):

```verilog
assign input_pu  = '0;
assign input_pd  = '0;
assign bidir_oe  = '1;    // bidir pads drive outwards
assign bidir_cs  = '0;    // CMOS buffer (not Schmitt)
assign bidir_sl  = '0;    // fast slew
assign bidir_ie  = ~bidir_oe;
assign bidir_pu  = '0;
assign bidir_pd  = '0;
```

The only prerequisite is a Docker host with the
`hpretl/iic-osic-tools:next` image and an already-prepared
`chipathon_padring/template/` checkout on disk. Runtime of one
full pass: ~35-45 min on the reference machine (can be many hours
if the host is slow; Magic DRC dominates).

## Step 0 — configuration

In [ ]:
from pathlib import Path
import os
import subprocess
import textwrap

# ---- flags ----
RUN_WRITE_CORE = False   # writes your chip_core.sv into the template
RUN_FLOW       = False   # ~35-45 min; dominated by Magic DRC

# ---- container (must already be running with the workspace mount) ----
CONTAINER_NAME = "gf180"

# ---- host paths ----
HOST_WORKSPACE  = Path.home() / "eda" / "designs"
HOST_TEMPLATE   = HOST_WORKSPACE / "chipathon_padring" / "template"
CORE_PATH       = HOST_TEMPLATE / "src" / "chip_core.sv"

# ---- container paths ----
CONTAINER_TEMPLATE = "/foss/designs/chipathon_padring/template"

# ---- PDK identifiers ----
PDK_NAME     = "gf180mcuD"
STD_CELL_LIB = "gf180mcu_fd_sc_mcu7t5v0"


def run_or_print(cmd, do_it, *, shell_on_container=False, timeout=None):
    if shell_on_container:
        print(f"$ docker exec {CONTAINER_NAME} bash -lc '<script>'")
        print(textwrap.indent(cmd, "  | "))
    else:
        print("$ " + " ".join(cmd))
    if not do_it:
        print("  (skipped -- flip the RUN_* flag to execute)\n")
        return None
    args = (
        ["docker", "exec", CONTAINER_NAME, "bash", "-lc", cmd]
        if shell_on_container else cmd
    )
    proc = subprocess.run(args, capture_output=True, text=True, timeout=timeout)
    if proc.stdout.strip():
        print(proc.stdout[-4000:])
    if proc.returncode != 0 and proc.stderr.strip():
        print("STDERR (tail):")
        print(proc.stderr[-2000:])
    print(f"  returncode={proc.returncode}\n")
    return proc


print(f"Template (host):      {HOST_TEMPLATE}")
print(f"chip_core.sv path:    {CORE_PATH}")
print(f"Template (container): {CONTAINER_TEMPLATE}")

## Step 1 — verify the template is ready

Sanity check: the workshop slot exists, the wafer-space PDK fork is
in place, and the container is alive. If anything is missing, the
cell below tells you what to fix (most likely: run
`rtl2gds_chipathon_padring.ipynb` first, or extract the chipathon
tarball into `~/eda/designs/chipathon_padring/`).

In [ ]:
def ok(label, cond, detail=""):
    tag = "OK " if cond else "!! "
    print(f"{tag}{label}" + (f"  -- {detail}" if detail else ""))
    return cond

all_ok = True
all_ok &= ok(f"Template dir exists", HOST_TEMPLATE.exists(), str(HOST_TEMPLATE))
all_ok &= ok("slot_workshop.yaml present",
             (HOST_TEMPLATE / "librelane" / "slots" / "slot_workshop.yaml").exists())
all_ok &= ok("SLOT_WORKSHOP block in slot_defines.svh",
             "SLOT_WORKSHOP" in (HOST_TEMPLATE / "src" / "slot_defines.svh").read_text()
             if (HOST_TEMPLATE / "src" / "slot_defines.svh").exists() else False)
all_ok &= ok("wafer-space PDK fork present",
             (HOST_TEMPLATE / "gf180mcu" / "gf180mcuD").exists())
all_ok &= ok("Makefile lists 'workshop' in AVAILABLE_SLOTS",
             "workshop" in (HOST_TEMPLATE / "Makefile").read_text()
             if (HOST_TEMPLATE / "Makefile").exists() else False)

proc = subprocess.run(
    ["docker", "ps", "--filter", f"name={CONTAINER_NAME}", "--format", "{{.Names}}"],
    capture_output=True, text=True,
)
all_ok &= ok(f"Container '{CONTAINER_NAME}' running", CONTAINER_NAME in proc.stdout)

if not all_ok:
    print("\n Fix the missing items before proceeding. Most likely you "
          "need to run rtl2gds_chipathon_padring.ipynb once, or extract "
          "the chipathon template tarball into ~/eda/designs/chipathon_padring/.")

## Step 2 — write your `chip_core.sv`

The default example below implements a **visible counter** whose
state is exposed on the 20 digital config pads, plus a trivial
passthrough that keeps the 60 analog pads usable (they are left
floating inside the core so a scope probe on any `ana<N>` pin can
observe the external bondpad directly).

Replace the `YOUR CORE LOGIC` section with your own RTL. Rules:

1. Keep the port list identical to what is shown. The parameters
   `NUM_INPUT_PADS` / `NUM_BIDIR_PADS` / `NUM_ANALOG_PADS` will be
   bound by `chip_top.sv` to `{1, 20, 60}` for the workshop slot.
2. Drive every output (`input_pu`, `input_pd`, `bidir_out`,
   `bidir_oe`, ...). If you forget one, Yosys halts the flow.
3. Do not instantiate the pad cells yourself; `chip_top.sv` handles
   that.
4. If you add your own macros (SRAM, analog IP), you also need to
   patch `librelane/config.yaml` to register them under `MACROS:` --
   that goes beyond this notebook; see
   `rtl2gds_chip_top_custom.ipynb` for an example of a custom macro.

In [ ]:
CHIP_CORE_USER = '''\
// Chipathon 2026 -- your custom chip_core for the workshop padring.
// Replace the YOUR CORE LOGIC block with your own RTL.

`default_nettype none

module chip_core #(
    parameter NUM_INPUT_PADS,
    parameter NUM_BIDIR_PADS,
    parameter NUM_ANALOG_PADS
    )(
    `ifdef USE_POWER_PINS
    inout  wire VDD,
    inout  wire VSS,
    `endif

    input  wire clk,
    input  wire rst_n,

    input  wire [NUM_INPUT_PADS-1:0] input_in,
    output wire [NUM_INPUT_PADS-1:0] input_pu,
    output wire [NUM_INPUT_PADS-1:0] input_pd,

    input  wire [NUM_BIDIR_PADS-1:0] bidir_in,
    output wire [NUM_BIDIR_PADS-1:0] bidir_out,
    output wire [NUM_BIDIR_PADS-1:0] bidir_oe,
    output wire [NUM_BIDIR_PADS-1:0] bidir_cs,
    output wire [NUM_BIDIR_PADS-1:0] bidir_sl,
    output wire [NUM_BIDIR_PADS-1:0] bidir_ie,
    output wire [NUM_BIDIR_PADS-1:0] bidir_pu,
    output wire [NUM_BIDIR_PADS-1:0] bidir_pd,

    inout  wire [NUM_ANALOG_PADS-1:0] analog
);

    // ---- default-safe pad controls (keep these unless you know why you don't) ----
    assign input_pu = '0;
    assign input_pd = '0;
    assign bidir_oe = '1;    // drive outwards
    assign bidir_cs = '0;    // CMOS buffer (not Schmitt)
    assign bidir_sl = '0;    // fast slew
    assign bidir_ie = ~bidir_oe;
    assign bidir_pu = '0;
    assign bidir_pd = '0;

    // Keep synthesis from dropping un-connected inputs -- harmless.
    logic _unused;
    assign _unused = &{1'b0, bidir_in, input_in};

    // ---- YOUR CORE LOGIC STARTS HERE ----
    // Example: a 20-bit counter exposed on the 20 digital bidir pads.
    // Bits roll every ~0.5 s at 1 MHz input clock.  Replace with your RTL.
    logic [NUM_BIDIR_PADS-1:0] count;
    always_ff @(posedge clk) begin
        if (!rst_n) count <= '0;
        else        count <= count + 1;
    end
    assign bidir_out = count;
    // ---- YOUR CORE LOGIC ENDS HERE ----

    // The 60 analog pads stay untouched at the core level; bring them
    // to internal analog IP by naming each explicitly if needed, e.g.:
    //   my_opamp u_op (.vp (analog[0]), .vn (analog[1]), .out (analog[2]));

endmodule

`default_nettype wire
'''

if RUN_WRITE_CORE:
    CORE_PATH.write_text(CHIP_CORE_USER)
    print(f"Wrote {CORE_PATH}  ({len(CHIP_CORE_USER.splitlines())} lines)")
else:
    print("Preview of chip_core.sv (flip RUN_WRITE_CORE = True to commit to disk):\n")
    print(textwrap.indent(CHIP_CORE_USER, "    "))

## Step 3 — run the flow

```bash
# Inside the container:
cd /foss/designs/chipathon_padring/template
source sak-pdk-script.sh gf180mcuD gf180mcu_fd_sc_mcu7t5v0
make librelane SLOT=workshop PDK=gf180mcuD PDK_ROOT=./gf180mcu
```

Expected wall time: 35-45 min on a fast host, multiple hours if
Magic DRC is slow. `RUN_FLOW` stays `False` by default so the
cell can rehearse the command without executing it.

In [ ]:
flow_script = textwrap.dedent(f"""
    set -e
    cd {CONTAINER_TEMPLATE}
    source sak-pdk-script.sh {PDK_NAME} {STD_CELL_LIB}
    make librelane \\
        SLOT=workshop \\
        PDK={PDK_NAME} \\
        PDK_ROOT={CONTAINER_TEMPLATE}/gf180mcu
""").strip()

# Generous timeout so Magic DRC can finish on slow hosts.
run_or_print(flow_script, RUN_FLOW, shell_on_container=True, timeout=43200)

## Step 4 — read `metrics.csv`

One row per metric in `final/metrics.csv`. A green chip has zeros
across every violation counter.

In [ ]:
import csv

metrics_path = HOST_TEMPLATE / "final" / "metrics.csv"
wanted = [
    "design__die__area__um2",
    "design__instance__count",
    "design__instance__count__stdcell",
    "design__instance__count__class:macro",
    "magic__drc_error__count",
    "klayout__drc_error__count",
    "design__lvs_error__count",
    "antenna__violating__nets",
    "timing__setup_vio__count",
    "timing__hold_vio__count",
    "power__total",
]

if not metrics_path.exists():
    print(f"metrics.csv not found: {metrics_path}")
    print("Did Step 3 complete?  Set RUN_FLOW = True and re-run.")
else:
    found = {}
    with metrics_path.open() as fh:
        for row in csv.reader(fh):
            if row and row[0] in wanted:
                found[row[0]] = row[1] if len(row) > 1 else ""
    for k in wanted:
        print(f"  {k:45s} {found.get(k, '(missing)')}")

## Step 5 — show the render

The final layout is at `final/gds/chip_top.gds`. LibreLane's
`KLayout.Render` step writes `final/render/chip_top.png` for
quick visual inspection.

In [ ]:
from IPython.display import Image, display

png_path = HOST_TEMPLATE / "final" / "render" / "chip_top.png"
gds_path = HOST_TEMPLATE / "final" / "gds"    / "chip_top.gds"

if png_path.exists():
    print(f"Render: {png_path}")
    display(Image(str(png_path)))
else:
    print(f"No render PNG yet at {png_path}")
    if gds_path.exists():
        print(f"GDS is there: {gds_path}")
        print(f"  open on the host:  klayout {gds_path}")

## Where to go next

1. **Add custom macros** (pre-hardened block -- SRAM, opamp, SAR ADC
   DAC). Harden the block on its own with the Classic flow
   (`rtl2gds_counter.ipynb` as a template), then reference it under
   `MACROS:` in `librelane/config.yaml`. See
   `rtl2gds_chip_top_custom.ipynb` for a worked counter example.
2. **Connect analog IP** to `analog[N]` pins. The 60 analog pads in
   the workshop padring are `gf180mcu_fd_io__asig_5p0` cells; they
   tolerate the 5 V supply domain of the pad ring.
3. **Verify with cocotb** before running the heavy flow. The
   template has `make verify SLOT=workshop`.
4. **Tape out**. The workshop padring matches the shuttle's
   2935x2935 um tile; once DRC/LVS/timing are green, your
   `final/gds/chip_top.gds` is ready to hand in.

Clean up:

```bash
rm -rf ~/eda/designs/chipathon_padring/template/librelane/runs
docker stop gf180
```